<a href="https://colab.research.google.com/github/Atharvasw2005/SmartAlignOra_Software/blob/Machine-Learning/SmartAlignOra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

WINDOW_SIZE = 20
features = []
labels = []

# -----------------------
# Load CSV Files
# -----------------------
for file in os.listdir("/content"):
    if file.endswith(".csv"):
        df = pd.read_csv("/content/" + file)

        pitch = df["pitch"].values

        if "good" in file.lower():
            label = 1
        else:
            label = 0

        for i in range(len(pitch) - WINDOW_SIZE):
            window = pitch[i:i+WINDOW_SIZE]

            feat = [
                np.mean(window),
                np.std(window),
                np.min(window),
                np.max(window),
                np.max(window) - np.min(window)
            ]

            features.append(feat)
            labels.append(label)

X = np.array(features)
y = np.array(labels)

# -----------------------
# Train Test Split
# -----------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------
# Train Model
# -----------------------
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# -----------------------
# Evaluate
# -----------------------
preds = model.predict(X_test)
accuracy = accuracy_score(y_test, preds)
print("Accuracy:", accuracy)

# -----------------------
# Save Model
# -----------------------
joblib.dump(model, "/content/posture_model.pkl")
print("Model Saved!")


Accuracy: 0.9596577017114915
Model Saved!


In [11]:
import joblib
import numpy as np

model = joblib.load("/content/posture_model.pkl")

sample_window = [

-0.02, -0.05, -0.06, -0.04, -0.03,
-0.01, -0.05, -0.06, -0.07, -0.04,
-0.05, -0.06, -0.04, -0.03, -0.05,
-0.04, -0.06, -0.05, -0.04, -0.03


]

features = [
    np.mean(sample_window),
    np.std(sample_window),
    np.min(sample_window),
    np.max(sample_window),
    max(sample_window) - min(sample_window)
]

pred = model.predict([features])[0]
proba = model.predict_proba([features])[0]
confidence = max(proba)

label = "GOOD" if pred == 1 else "BAD"
print("Prediction:", label)

print("Confidence:", confidence)
THRESHOLD = 0.8

if pred == 0 and confidence >= THRESHOLD:
    print("ALERT: BAD POSTURE")
elif pred == 1 and confidence >= THRESHOLD:
    print("POSTURE OK")
else:
    print("UNCERTAIN - IGNORE")



Prediction: GOOD
Confidence: 0.83
POSTURE OK
